In [1]:
import sys
print(sys.executable)

/usr/bin/python3


In [2]:
!pip install -q -U peft
!nvidia-smi

Fri Nov  6 08:55:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   74C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install -q transformers

In [4]:
!pip install -q -U "bitsandbytes>=0.46.1"

In [5]:
!pip install torch==2.11.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

Looking in indexes: https://download.pytorch.org/whl/cu128


In [6]:
pip install -q -U torchao

Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [8]:
import torch
import numpy as np
import pandas as pd
import os
print("Pytorch version:", torch.__version__)
print("CUDA AVAILABLE:", torch.cuda.is_available())
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

Pytorch version: 2.11.0+cu130
CUDA AVAILABLE: True


In [9]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("HF_TOKEN")


login(token=token)

## Loading model and tokenizer

In [34]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
from peft import PeftModel

base_model_id = "facebook/nllb-200-distilled-1.3B"
adapter_model_id = "coded-by-49/LANGBRIDGE_AI_IGBO_1.0"
adapter_model_id_2 = "coded-by-49/LANG_BRIDGE_AI_2.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float32,
)


tokenizer = AutoTokenizer.from_pretrained(
    adapter_model_id_2,
    #variables that needs to be changed based on on evaluation
    src_lang="eng_Latn",
    tgt_lang="ibo_Latn",
)

base_model =  AutoModelForSeq2SeqLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    dtype=torch.float32,
    device_map = "auto",
)

langbride_ai_igbo = PeftModel.from_pretrained(
    base_model,
    adapter_model_id,
)

langbride_ai_v2 = PeftModel.from_pretrained(
    base_model,
    adapter_model_id_2,
)


Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

## Importing test dataset

In [35]:
from datasets import load_dataset
import random

test_dataset = load_dataset(
    "coded-by-49/langbridge_ai_v3_igbo_datasets",
    split = "test"
)


In [36]:
src_eng_sentences = [item["eng_Latn"] for item in test_dataset["translation"]]
src_ibo_sentences = [item["ibo_Latn"] for item in test_dataset["translation"]]

In [37]:
print(src_eng_sentences[10])
print(src_ibo_sentences[10])

people are after the 2018 budget of Nigeria and why it has not been published.
Ndị mmadu na-ajụ maka akwụkwọ mkpata na mmefu ego nke afọ 2018 nke Naịjirịa na ihe mere ejibeghi wepụta ya.


## Tokenized dataset class

In [38]:
from torch.utils.data import Dataset

class TokenizedDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, index):
        return {key: value[index] for key, value in self.encodings.items()}

## Tokenizing the full test datasets

In [39]:
ibo_lang_encodings = tokenizer(
    src_ibo_sentences,
    truncation=True,
    padding = False,
    # return_tensors = "pt",
    max_length = 256,
)
full_tokenized_ibo_dataset = TokenizedDataset(ibo_lang_encodings)
print(f"Full tokenized dataset size: {len(full_tokenized_ibo_dataset)}")

Full tokenized dataset size: 1693


In [40]:
eng_lang_encodings = tokenizer(
    src_eng_sentences,
    truncation=True,
    padding = False,
    # return_tensors = "pt",
    max_length = 256,
)
full_tokenized_eng_dataset = TokenizedDataset(eng_lang_encodings)
print(f"Full tokenized dataset size: {len(full_tokenized_eng_dataset)}")

Full tokenized dataset size: 1693


In [41]:
print(full_tokenized_eng_dataset[0:2])

{'input_ids': [[256047, 46280, 209967, 10833, 1482, 19553, 15807, 80778, 2125, 75007, 22730, 248079, 6370, 16300, 280, 92441, 123123, 5057, 7197, 248075, 2], [256047, 1808, 10833, 1482, 248079, 452, 349, 100424, 69796, 452, 349, 190720, 452, 106329, 107, 248080, 248079, 1482, 9872, 10275, 2442, 202, 59533, 502, 281, 349, 68920, 452, 114233, 36901, 100481, 2442, 351, 10894, 237025, 540, 1836, 49211, 439, 16346, 27523, 16701, 1482, 38406, 202, 7527, 49232, 1482, 4039, 248079, 208, 59533, 502, 349, 9872, 171418, 32563, 31253, 80778, 1482, 8444, 23988, 29950, 202, 78537, 905, 29, 28779, 76286, 384, 1482, 193155, 44778, 392, 248079, 115981, 135277, 8444, 148327, 108, 349, 33407, 47969, 248075, 2]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

## Data collator

In [42]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=langbride_ai_v2,
    padding=True,
    pad_to_multiple_of = 8,
)

## ⚙️ Evaluation Configuration
This is the **only cell you need to change** between runs.
- `EVAL_START` and `EVAL_END` define the total window from the 2000-sample pool
- `CHUNK_SIZE` must stay at 100 (GPU memory limit)
- `LANG_PAIR` is used to name the output file

In [43]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
EVAL_START  = 0     # start index into the 2000-sample pool
EVAL_END    = 1693  # end index (ceiling). Must be <= 2000
CHUNK_SIZE  = 100  # keep at 100 — GPU OOMs above this
LANG_PAIR   = "eng_ibo"  # used to name the output file
OUTPUT_PATH = f"lb_v2_{LANG_PAIR}_comet_eval_data_{EVAL_START}_{EVAL_END}_main.json"
# ────────────────────────────────────────────────────────────────────────────

assert EVAL_END <= 2000, "EVAL_END cannot exceed the 2000-sample pool size"
assert EVAL_START < EVAL_END, "EVAL_START must be less than EVAL_END"
assert CHUNK_SIZE <= 100, "CHUNK_SIZE above 100 risks GPU OOM"

total_samples = EVAL_END - EVAL_START
num_chunks = (total_samples + CHUNK_SIZE - 1) // CHUNK_SIZE
print(f"Evaluating indices [{EVAL_START}, {EVAL_END}) → {total_samples} samples across {num_chunks} chunks")
print(f"Output → {OUTPUT_PATH}")

Evaluating indices [0, 1693) → 1693 samples across 17 chunks
Output → lb_v2_eng_ibo_comet_eval_data_0_1693_main.json


## 🔁 Automated Chunked Inference Loop
Iterates over the evaluation window in steps of `CHUNK_SIZE`.
Each chunk appends `{src, mt, ref}` dicts to `comet_eval_data`.
No manual re-running or copy-pasting needed.

In [44]:
import gc
import json
from torch.utils.data import DataLoader, Subset

lang_translated_to = "ibo_Latn"
comet_eval_data = []  # accumulator: list of {src, mt, ref} dicts

langbride_ai_igbo.eval()

for chunk_idx, chunk_start in enumerate(range(EVAL_START, EVAL_END, CHUNK_SIZE)):
    chunk_end = min(chunk_start + CHUNK_SIZE, EVAL_END)
    print(f"\n── Chunk {chunk_idx + 1}/{num_chunks}: indices [{chunk_start}, {chunk_end}) ──")

    # For the model (tokenized dicts) → DataLoader
    chunk_src_dataset = Subset(full_tokenized_eng_dataset, range(chunk_start, chunk_end))

    # For COMET (plain text)(this would be changed when making )
    chunk_src_text = src_ibo_sentences[chunk_start:chunk_end]
    chunk_ref_text = src_eng_sentences[chunk_start:chunk_end]

    chunk_loader = DataLoader(
        chunk_src_dataset,
        batch_size=8,
        shuffle=False,
        num_workers=4,
        collate_fn=data_collator,
    )

    if len(chunk_src_dataset) == 0:
        print(f"Chunk {chunk_idx + 1} is empty — skipping")
        continue

    # ── Inference ────────────────────────────────────────────────────────────
    chunk_output_ids = []

    with torch.no_grad():
        for batch in chunk_loader:
            input_ids = batch["input_ids"].to(langbride_ai_v2.device)
            attention_mask = batch["attention_mask"].to(langbride_ai_v2.device)
            langbride_ai_v2.generation_config.max_length = None
            generated_ids = langbride_ai_v2.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=256,
                num_beams=4,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids(lang_translated_to),
                no_repeat_ngram_size=3,
            )

            chunk_output_ids.extend(generated_ids)

            del input_ids, attention_mask, generated_ids
            gc.collect()
            torch.cuda.empty_cache()

    # ── Decode ───────────────────────────────────────────────────────────────
    chunk_mt_texts = tokenizer.batch_decode(chunk_output_ids, skip_special_tokens=True)

    # ── Concordance check ────────────────────────────────────────────────────
    assert len(chunk_src_text) == len(chunk_mt_texts) == len(chunk_ref_text), (
        f"Chunk {chunk_idx + 1}: length mismatch — "
        f"src={len(chunk_src_text)}, mt={len(chunk_mt_texts)}, ref={len(chunk_ref_text)}"
    )

    # ── Accumulate ───────────────────────────────────────────────────────────
    for src, mt, ref in zip(chunk_src_text, chunk_mt_texts, chunk_ref_text):
        comet_eval_data.append({"src": src, "mt": mt, "ref": ref})

    print(f"   ✓ Accumulated {len(comet_eval_data)} total samples so far")

    # ── Per-chunk memory cleanup ──────────────────────────────────────────────
    del chunk_output_ids, chunk_mt_texts, chunk_src_text, chunk_ref_text, chunk_loader
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n✅ Inference complete — {len(comet_eval_data)} samples accumulated")


── Chunk 1/17: indices [0, 100) ──
   ✓ Accumulated 100 total samples so far

── Chunk 2/17: indices [100, 200) ──
   ✓ Accumulated 200 total samples so far

── Chunk 3/17: indices [200, 300) ──
   ✓ Accumulated 300 total samples so far

── Chunk 4/17: indices [300, 400) ──
   ✓ Accumulated 400 total samples so far

── Chunk 5/17: indices [400, 500) ──
   ✓ Accumulated 500 total samples so far

── Chunk 6/17: indices [500, 600) ──
   ✓ Accumulated 600 total samples so far

── Chunk 7/17: indices [600, 700) ──
   ✓ Accumulated 700 total samples so far

── Chunk 8/17: indices [700, 800) ──
   ✓ Accumulated 800 total samples so far

── Chunk 9/17: indices [800, 900) ──
   ✓ Accumulated 900 total samples so far

── Chunk 10/17: indices [900, 1000) ──
   ✓ Accumulated 1000 total samples so far

── Chunk 11/17: indices [1000, 1100) ──
   ✓ Accumulated 1100 total samples so far

── Chunk 12/17: indices [1100, 1200) ──
   ✓ Accumulated 1200 total samples so far

── Chunk 13/17: indices [1200,

## 💾 Writing AfriCOMET evaluation data to file
Writes the full accumulator as a JSON array, then clears it from memory.

In [45]:
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(comet_eval_data, f, ensure_ascii=False, indent=4)

print(f"✅ Saved {len(comet_eval_data)} samples → {OUTPUT_PATH}")

# Verify the file round-trips correctly
with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    verification = json.load(f)

print(f"✅ Verification: {len(verification)} records readable from file")
print(f"   First record: {verification[0]}")
print(f"   Last  record: {verification[-1]}")

# Clear accumulator from memory
del comet_eval_data, verification
gc.collect()
print("\n🧹 Accumulator cleared from memory")

✅ Saved 1693 samples → lb_v2_eng_ibo_comet_eval_data_0_1693_main.json
✅ Verification: 1693 records readable from file
   First record: {'src': 'Adeyeye kwukwara na imepe ụlọọrụ ndị a ga-agbado ụkwụ ka ụlọọrụ ndị a si nye Nafac nkwado na nyocha ha.', 'mt': "Adeyeye kwuru na tupu ụlọ ọrụ ndị a enwee ike ịrụ ọrụ ọzọ, a ga-enwerịrị nkwado kachasị site n'aka ha.", 'ref': 'Adeyeye said that before these companies can function again, there must be maximum cooperation from them.'}
   Last  record: {'src': "Onye Mgbazi, ACTDA, Ven. Amaechi Okwuosa kwuru na ndị isi, n'okpuru ọchịchị gọvanọ bụ ndị nke enyere ikike iwepụta usoro ukwu nke Ọka, igbanwe isi obodo ka ọ bụrụ Dubai ma ọbụ Taiwan", 'mt': "Onye isi nchịkwa, ACTDA, Ven. Amaechi Okwuosa kwuru na e nyere ikike, n'okpuru ụlọ ọrụ gọvanọ, ọrụ ịkwadebe atụmatụ nke Awka, na-atụgharị isi obodo na Dubai ma ọ bụ Taiwan.", 'ref': 'Managing Director, ACTDA, Ven. Amaechi Okwuosa said that the Authority, under the office of the governor was charged with